# OpenSLA Demo

This notebook demonstrates how to:
1. Load a pretrained OpenSLA-B / OpenSLA-H checkpoint
2. Build an input batch (your own preprocessed batch, or a synthetic one)
3. Run action prediction and caption generation
4. Read the structured outputs

In [ ]:
import json
from pathlib import Path
import torch
from opensla import OpenSLA, SensorBatch, load_prepared_batch, make_synthetic_batch

## Configuration

`configs/template.json` points to the checkpoint, the waveform-encoder checkpoint, the numeric configuration, and the input batch. The input format is described in the README.

In [ ]:
config = json.loads(Path("configs/template.json").read_text())
DEVICE = config["device"] if torch.cuda.is_available() else "cpu"
config

## 1. Load the pretrained model

The variant (B or H), the architecture settings, and the action vocabulary come from the checkpoint.

In [ ]:
model = OpenSLA.from_checkpoint(
    config["checkpoint"], domain=config["domain"], device=DEVICE,
    model_id=config.get("model_id"),
    dino_checkpoint=config.get("dino_checkpoint"),
    numeric_config=config.get("numeric_config"),
    action_group_types=config.get("action_group_types"),
)
print(model.name, "| domain:", config["domain"], "| action categories:", model.category_names)

## 2. Build an input batch

Use your own preprocessed `.pt` batch. If there is none, `make_synthetic_batch` builds a random batch with the right shapes so you can try the pipeline.

In [ ]:
if Path(config["input"]).exists():
    batch = load_prepared_batch(config["input"], domain=config["domain"])
    print("Loaded", len(batch["input_text"]), "samples from", config["input"])
else:
    batch = make_synthetic_batch(model, batch_size=2)
    print("Using a synthetic batch of", len(batch["input_text"]), "samples")

sensors = batch["sensors"]
if sensors.waveform is not None:
    print("waveform", tuple(sensors.waveform.shape), "channel_mask", tuple(sensors.waveform_channel_mask.shape))
if sensors.numeric is not None:
    print({k: tuple(v.shape) for k, v in sensors.numeric.items()})

## 3. Predict

`max_new_tokens=0` runs the action heads only; a positive value also generates the caption.

In [ ]:
predictions = model.predict(
    batch["input_text"], sensors,
    max_new_tokens=config["max_new_tokens"],
    necessity_threshold=config["necessity_threshold"],
    category_threshold=config["category_threshold"],
    labels_per_category=config["labels_per_category"],
    max_categories=config["max_categories"],
)

## 4. Read the outputs

Each prediction contains the necessity probability, per-category probabilities, the selected categories, ranked fine labels (cosine similarities), and the generated caption.

In [ ]:
for sample_id, result in zip(batch["sample_ids"], predictions):
    print("==", sample_id)
    print(json.dumps(result, indent=2))